# Pre procesamiento trazas de movilidad

## Configuraciones iniciales

In [ ]:
# ── Configuración del repositorio ──────────────────────────────────────────
# Este cuaderno implementa los Módulos I y II del pipeline. Opera sobre
# cualquier fuente de telefonía móvil transformable al esquema canónico
# declarado en config.COLUMN_MAP (ver Tabla 3.1 del manuscrito).
#
# La fuente utilizada en el proyecto es confidencial y no forma parte de esta
# entrega. Para ejecutar el cuaderno, complete COLUMN_MAP con los nombres de
# campo de su propia fuente y apunte config.TELCO_PARQUET_DIR a su ubicación.

import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
import config as cfg

print(cfg.describe_config())

In [ ]:
# Importando librerías útiles
import gc, os
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
from tqdm.auto import tqdm
from IPython.display import display

from geopy.distance import geodesic
from shapely.ops import nearest_points
from pyproj import Geod
import pyarrow
import pyogrio

### Versiones de las librerías utilizadas

A continuación se detallan las versiones de las librerías empleadas en el presente código. Las pertenecientes a la biblioteca estándar se indican con “(stdlib)” y muestran la versión de Python del entorno.

| package          | version       |
| ---------------- | ------------- |
| numpy            | 1.24.4        |
| pandas           | 2.0.3         |
| geopandas        | 0.13.2        |
| tqdm             | 4.67.1        |
| IPython          | 8.12.3        |
| geopy            | 2.4.1         |
| shapely          | 2.0.7         |
| pyproj           | 3.5.0         |
| pyarrow          | 17.0.0        |
| pyogrio          | 0.9.0         |
| gc (stdlib)      | Python 3.8.10 |
| os (stdlib)      | Python 3.8.10 |
| pathlib (stdlib) | Python 3.8.10 |

## Lectura de datos

In [ ]:
# 1) Localizar los archivos de la fuente telco
cfg.require(cfg.TELCO_PARQUET_DIR, "el directorio de la fuente telco", nivel=2)
parquet_files = sorted(cfg.TELCO_PARQUET_DIR.glob(cfg.TELCO_PARQUET_GLOB))
print(f"Archivos Parquet detectados: {len(parquet_files)}")

# 2) Lectura
dfs = []
for file in tqdm(parquet_files, desc="Leyendo archivos"):
    dfs.append(pd.read_parquet(file, engine="pyarrow"))

df_total = pd.concat(dfs, ignore_index=True)

# 3) Estandarización de esquema (Módulo I): traducción al esquema canónico
df_total = cfg.apply_column_map(df_total)

del dfs
_ = gc.collect()

print("\nShape total:", df_total.shape)
print("Esquema canónico:", list(df_total.columns))
df_total.head()

## Tratamiento de duplicados

### Duplicados exactos

In [ ]:
before = len(df_total)
df_total.drop_duplicates(inplace=True)
after = len(df_total)

print(f"Duplicados eliminados: {before - after}")
print(f"Shape final: {df_total.shape}")

### Usuario anómalo

In [ ]:
# Exclusión de identificadores incompatibles con movilidad humana ordinaria.
# La regla pertenece al pipeline; los valores dependen de cada fuente y deben
# justificarse a partir de la distribución empírica de registros por usuario.
if cfg.ANOMALOUS_USER_IDS:
    df_filtrado = df_total[
        ~df_total[cfg.COL_USER].isin(cfg.ANOMALOUS_USER_IDS)
    ].copy()
    print(f"Identificadores excluidos: {len(cfg.ANOMALOUS_USER_IDS)}")
    print(f"Registros eliminados: {df_total.shape[0] - df_filtrado.shape[0]}")
else:
    df_filtrado = df_total.copy()
    print(
        "ANOMALOUS_USER_IDS está vacío: no se excluyó ningún identificador.\n"
        "Inspeccione la distribución de registros por usuario y complete la "
        "lista en config.py si corresponde."
    )

print(f"Shape resultante: {df_filtrado.shape}")

### Duplicados par (user, timestamps)

> **Duplicado user + timestamp**: el par (`user_id_anon`, `event_ts`) se repite.
> Se considera duplicado bajo el supuesto de que **un usuario no puede estar en dos ubicaciones distintas al mismo tiempo**.

In [ ]:
# ── 0) Marcar duplicados tipo 2 (usuario + timestamp) en df_filtrado
duplicados_par = df_filtrado.duplicated(subset=[cfg.COL_USER, cfg.COL_TS], keep=False)

# ── 1) Filtrar registros duplicados tipo 2 en df_filtrado ──────
df_dupl = df_filtrado[duplicados_par].copy()

# ── 2) Agrupar por (usuario, timestamp) ────────────────────────
grupos = df_dupl.groupby([cfg.COL_USER, cfg.COL_TS])

# ── 3) Calcular distancia geodésica entre registros duplicados ─
distancias = []
errores = 0

for _, group in grupos:
    if len(group) == 2:
        coords = list(zip(group[cfg.COL_LAT], group[cfg.COL_LON]))
        dist = geodesic(coords[0], coords[1]).meters
        distancias.append(dist)
    else:
        errores += 1  # casos con más o menos de 2 registros

# ── 4) Estadísticas descriptivas ───────────────────────────────
dist_array = np.array(distancias)

print(f"Cantidad de pares analizados: {len(dist_array):,}")
print(f"Errores (no pares de 2 registros): {errores}")
print(f"Media: {dist_array.mean():.2f} m")
print(f"Mediana: {np.median(dist_array):.2f} m")
print(f"Máximo: {dist_array.max():.2f} m")
print(f"P10: {np.percentile(dist_array, 10):.2f} m")
print(f"P20: {np.percentile(dist_array, 20):.2f} m")
print(f"P30: {np.percentile(dist_array, 30):.2f} m")
print(f"P40: {np.percentile(dist_array, 40):.2f} m")
print(f"P50: {np.percentile(dist_array, 50):.2f} m")
print(f"P60: {np.percentile(dist_array, 60):.2f} m")
print(f"P70: {np.percentile(dist_array, 70):.2f} m")
print(f"P80: {np.percentile(dist_array, 80):.2f} m")
print(f"P90: {np.percentile(dist_array, 90):.2f} m")
print(f"P95: {np.percentile(dist_array, 95):.2f} m")

In [ ]:
# 1. Agrupar por par (usuario, timestamp)
grupos = df_dupl.groupby([cfg.COL_USER, cfg.COL_TS])

# 2. Construir lista de distancias con información asociada
registros = []
for (usuario, ts), group in grupos:
    if len(group) == 2:
        coords = list(zip(group[cfg.COL_LAT], group[cfg.COL_LON]))
        dist = geodesic(coords[0], coords[1]).meters
        registros.append({
            "usuario": usuario,
            "timestamp": ts,
            "lat1": coords[0][0],
            "lon1": coords[0][1],
            "lat2": coords[1][0],
            "lon2": coords[1][1],
            "distancia_m": dist
        })

# 3. Convertir a DataFrame ordenado por distancia
df_distancias = pd.DataFrame(registros).sort_values("distancia_m").reset_index(drop=True)

# 4) Identificar pares con más de 2 registros (casos inválidos)
counts = df_dupl.groupby([cfg.COL_USER, cfg.COL_TS]).size()
pares_multiples = counts[counts > 2].index

# 5) Identificar pares separados por más del umbral de colisiones (DELTA_H_M)
pares_lejanos = df_distancias[df_distancias["distancia_m"] > cfg.DELTA_H_M][["usuario", "timestamp"]]
pares_lejanos = [tuple(x) for x in pares_lejanos.to_numpy()]

# 6) Unión de pares conflictivos
pares_a_eliminar = set(pares_multiples).union(set(pares_lejanos))

# 7) Ordenar para revisión
pares_a_eliminar = sorted(pares_a_eliminar)

# 8) Crear máscara para registros a eliminar
pares_a_eliminar_set = set(pares_a_eliminar)

# Máscara booleana: True si (usuario, timestamp) está en la lista
mask_conflictivos = df_filtrado[[cfg.COL_USER, cfg.COL_TS]].apply(
    lambda row: (row[cfg.COL_USER], row[cfg.COL_TS]) in pares_a_eliminar_set,
    axis=1
)

# 9) Aplicar eliminación
rows_before = len(df_filtrado)
df_filtrado = df_filtrado[~mask_conflictivos].copy()
df_filtrado.index = pd.RangeIndex(len(df_filtrado))  # compactar índice

print(f"Umbral de colisiones aplicado: {cfg.DELTA_H_M} m")
print(f"Registros antes: {rows_before:,}")
print(f"Eliminados: {mask_conflictivos.sum():,}")
print(f"Registros después: {len(df_filtrado):,}")

In [ ]:
# 1. Identificar duplicados tipo 2 válidos restantes
duplicados_validos = df_filtrado.duplicated(subset=[cfg.COL_USER, cfg.COL_TS], keep=False)
df_dupl_validos = df_filtrado[duplicados_validos].copy()

# 2. Agrupar por par (usuario, timestamp)
grupos = df_dupl_validos.groupby([cfg.COL_USER, cfg.COL_TS])

# 3. Fusionar cada par mediante el promedio aritmético de sus coordenadas
registros_fusionados = []
for (usuario, ts), group in grupos:
    if len(group) != 2:
        continue  # Por seguridad: ignoramos casos residuales
    lat_mean = group[cfg.COL_LAT].mean()
    lon_mean = group[cfg.COL_LON].mean()
    # Tomamos una fila base para mantener estructura de columnas
    row_base = group.iloc[0].copy()
    row_base[cfg.COL_LAT] = lat_mean
    row_base[cfg.COL_LON] = lon_mean
    registros_fusionados.append(row_base)

# 4. Crear nuevo DataFrame con registros fusionados
df_fusionados = pd.DataFrame(registros_fusionados)

# 5. Eliminar los registros originales duplicados tipo 2 del dataset
df_filtrado = df_filtrado[~duplicados_validos].copy()

# 6. Asegurar mismo formato datetime
df_fusionados[cfg.COL_TS] = pd.to_datetime(df_fusionados[cfg.COL_TS], errors="coerce").astype(df_filtrado[cfg.COL_TS].dtype)

# 7. Agregar los registros fusionados
df_filtrado = pd.concat([df_filtrado, df_fusionados], ignore_index=True)

# 8. Compactar índice final
df_filtrado.index = pd.RangeIndex(len(df_filtrado))

# 9. Verificación
print(f"Registros luego de fusionar duplicados tipo 2 válidos: {len(df_filtrado):,}")

# 10. Limpieza
del duplicados_validos, df_dupl_validos, grupos, registros_fusionados, df_fusionados
_ = gc.collect()

In [ ]:
# Renombrar dataframe final limpio
df_final = df_filtrado
del df_filtrado
_ = gc.collect()

# Marcar duplicados tipo 2 (usuario + timestamp) en df_final
duplicados_par = df_final.duplicated(subset=[cfg.COL_USER, cfg.COL_TS], keep=False)

# Contar total de registros involucrados en duplicados tipo 2
n_duplicados_par = duplicados_par.sum()

# Contar número de pares únicos duplicados
n_pares_duplicados = (
    df_final[duplicados_par][[cfg.COL_USER, cfg.COL_TS]]
    .drop_duplicates()
    .shape[0]
)

# Calcular porcentaje respecto al total del dataframe final
porcentaje_duplicados = (n_duplicados_par / len(df_final)) * 100

# Mostrar resultados
print(f"Registros con el mismo (usuario, timestamp): {n_duplicados_par}")
print(f"Pares únicos (usuario, timestamp) que se repiten: {n_pares_duplicados}")
print(f"Porcentaje del total: {porcentaje_duplicados:.4f}%")

Con estas operaciones el registro queda sin duplicados exactos ni colisiones simultáneas no resueltas en el par (user_id_anon, event_ts), conservando las fusiones compatibles con oscilación local entre antenas.

## Selección de horario válido

In [ ]:
def contar_registros_por_hora(df, col=cfg.COL_TS):
    """
    Devuelve DataFrame con:
      Hora (tramo HH:00–HH+1:00)
      Cantidad de registros
      Porcentaje sobre el total
    """
    # se asume que col ya es datetime64
    hour = df[col].dt.hour
    hourly_counts = hour.value_counts().sort_index()
    labels = [f"{h:02d}:00–{(h+1)%24:02d}:00" for h in hourly_counts.index]
    pct = (hourly_counts / len(df) * 100).round(3)

    return pd.DataFrame({
        "Hora": labels,
        "Cantidad de registros": hourly_counts.values,
        "Porcentaje (%)": pct.values
    })


# ── 1) Distribución inicial ────────────────────────────────────
print("Distribución horaria (antes del filtro):")
display(contar_registros_por_hora(df_final))

# ── 2) Registros por bloque horario ─────────────────────────────
h0, h1 = cfg.HOUR_WINDOW
hour = df_final[cfg.COL_TS].dt.hour
total_registros = len(df_final)

# Crear máscaras
mask_fuera_horario = (hour < h0) | (hour >= h1)
mask_dentro_horario = ~mask_fuera_horario

# Contar
fuera_count = mask_fuera_horario.sum()
dentro_count = mask_dentro_horario.sum()

# Mostrar
print("\nDistribución por bloque horario:")
print(f"  {h0:02d}:00–{h1:02d}:00 → {dentro_count:,} registros ({dentro_count / total_registros * 100:.2f} %)")
print(f"  {h1:02d}:00–{h0:02d}:00 → {fuera_count:,} registros ({fuera_count / total_registros * 100:.2f} %)")

# ── 3) Filtrar a la ventana horaria declarada, sin copia profunda ─────────
hour = df_final[cfg.COL_TS].dt.hour
mask_keep = (hour >= h0) & (hour < h1)

rows_before = len(df_final)
rows_remove = (~mask_keep).sum()

# Eliminación directa sin copia profunda
df_final.drop(index=df_final.index[~mask_keep], inplace=True)

# Compactar índice
df_final.index = pd.RangeIndex(len(df_final))

# Reporte
print(f"\nFilas antes: {rows_before:,}")
print(f"Eliminadas (fuera de [{h0:02d}:00, {h1:02d}:00)): {rows_remove:,}")
print(f"Filas después: {len(df_final):,}")

# ── 4) Distribución tras el filtro ─────────────────────────────
print("\nDistribución horaria (después del filtro):")
display(contar_registros_por_hora(df_final))

# Limpieza de variables temporales globales
del total_registros, mask_fuera_horario, mask_dentro_horario
del fuera_count, dentro_count, rows_before, rows_remove, mask_keep, hour, h0, h1
_ = gc.collect()

El registro queda restringido al dominio temporal declarado en config.HOUR_WINDOW. Las distribuciones anterior y posterior permiten verificar el efecto del filtro sobre el volumen retenido.

In [ ]:
df_final.head()

## Cruce con data geoespacial

In [ ]:
# 1) Cargar la DPA 2023
cfg.require(cfg.DPA_COMUNAS_SHP, "la cartografía DPA 2023", nivel=1)
gdf_comunas = gpd.read_file(cfg.DPA_COMUNAS_SHP, engine="pyogrio").to_crs(4326)
gdf_comunas.head()

In [ ]:
# 2) Observaciones -> GeoDataFrame en el CRS geográfico de trabajo
geom = gpd.points_from_xy(df_final.pop(cfg.COL_LON), df_final.pop(cfg.COL_LAT))
gdf_pings = gpd.GeoDataFrame(df_final, geometry=geom, crs=cfg.CRS_LATLON)
gdf_pings.head()

In [ ]:
gdf_pings.crs

In [ ]:
# 3) Spatial join
gdf_join = (
    gpd.sjoin(
        gdf_pings,     # observaciones telco en el CRS geográfico de trabajo
        gdf_comunas,   # comunas en el mismo CRS
        how="left",
        predicate="intersects"   # incluye los que caen sobre el límite
    )
    .drop(columns="index_right")  # limpia la columna auxiliar
)

del gdf_pings, geom
_ = gc.collect()

gdf_join.head()

In [ ]:
# 4) Drop redundantes (si existen)
cols_to_drop = [c for c in [cfg.COL_REGION,cfg.COL_COMUNA] if c in gdf_join.columns]
gdf_join.drop(columns=cols_to_drop, inplace=True)
gdf_join.head()

del cols_to_drop
_ = gc.collect()

In [ ]:
# 5) Nulos por columna
n_sin_comuna = gdf_join["CUT_COM"].isna().sum()
print(f"Puntos sin comuna tras el join: {n_sin_comuna:,}")

# (opcional) porcentaje
print(f"Porcentaje: {n_sin_comuna/len(gdf_join):.2%}")

In [ ]:
# ════════════════════════════════════════════════════════════════
# 7) SNAP IN-PLACE — corrección espacial mínima de observaciones
#    sin asignación territorial, dentro de la tolerancia DELTA_S_M.
#    Mecanismo residual, local y trazable (Módulo II).
# ════════════════════════════════════════════════════════════════

# ── 7-A. Sub-conjunto de pings sin comuna ──────────────────────
gdf_nan = gdf_join[gdf_join["CUT_COM"].isna()].copy()
print(f"Pings sin comuna antes del snap: {len(gdf_nan):,}")
assert gdf_nan.crs.to_epsg() == 4326, "GeoDataFrame debe estar en EPSG 4326"

In [ ]:
# 7-B-1) Renombrar columnas DPA ➜ sufijo _dpa
gdf_com_ren = gdf_comunas.rename(columns={
    c: f"{c}_dpa" for c in [
        "CUT_REG", "CUT_PROV", "CUT_COM",
        "REGION", "PROVINCIA", "COMUNA", "SUPERFICIE"
    ]
})
gdf_com_ren.head()

In [ ]:
# 7-B-2) CRS métrico para medición local de distancias (config.CRS_METRIC)
crs_metric = cfg.CRS_METRIC
gdf_nan_m = gdf_nan.to_crs(crs_metric)
gdf_com_m = gdf_com_ren.to_crs(crs_metric)

del gdf_com_ren
_ = gc.collect()

In [ ]:
# 7-B-3) sjoin_nearest dentro de la tolerancia declarada
gdf_near = gpd.sjoin_nearest(
    gdf_nan_m,
    gdf_com_m,
    how="left",
    max_distance=cfg.DELTA_S_M,   # tolerancia de corrección espacial, en metros
    distance_col="dist_m"
)

print(f"Tolerancia de snap aplicada: {cfg.DELTA_S_M} m")
print(f"Pings emparejados: {len(gdf_near):,}")
print(f"  • {gdf_near['CUT_COM_dpa'].notna().sum():,} pings encontraron comuna dentro de la tolerancia")
print(f"  • {gdf_near['CUT_COM_dpa'].isna().sum():,} pings siguen sin comuna")

del gdf_nan
_ = gc.collect()

In [ ]:
gdf_near.head()

In [ ]:
print("Columnas en gdf_near después del sjoin_nearest:")
print(gdf_near.columns)

In [ ]:
# ── 7-C. SNAP solo si hay pings por corregir ─────────────────────────────────
if gdf_near.empty:
    print(f"No hay pings fuera de la DPA dentro de {cfg.DELTA_S_M} m. Se omite el snap y se continúa.")
    # Deja columnas esperadas para etapas siguientes
    gdf_near["geometry_snap"] = pd.Series(dtype=object)  # vacío
else:
    # 7-C-0. Asignar geometría del polígono comunal (versión métrica)
    gdf_near["poly_geom"] = gdf_near["index_right"].map(gdf_com_m.geometry)

    # 7-C-1. Función de snap al interior del polígono
    def snap_inside(row):
        pt = row.geometry          # punto en el CRS métrico
        poly = row.poly_geom
        if poly is None or poly.contains(pt):
            return pt
        snapped, _ = nearest_points(poly, pt)   # cae sobre el borde
        if not poly.contains(snapped):
            inner = poly.buffer(-1)             # empuja 1 m al interior
            if not inner.is_empty:
                snapped, _ = nearest_points(inner, pt)
        return snapped

    # 7-C-2. Aplicar (asegurando Serie de objetos, no DataFrame)
    geom_snap = gdf_near.apply(snap_inside, axis=1)
    gdf_near = gdf_near.assign(geometry_snap=geom_snap.values)

    # Limpieza
    del snap_inside, geom_snap
    _ = gc.collect()

In [ ]:
gdf_near.crs

In [ ]:
# ════════════════════════════════════════════════════════════════
# 7-D · Volver al CRS geográfico + actualizar gdf_join
#      · Conservar geometría original en geometry_orig
#      · Copiar atributos comunales oficiales
# ════════════════════════════════════════════════════════════════

# 1) Reproyectar el resultado del snap al CRS geográfico de trabajo
crs_metric = gdf_near.crs  # CRS métrico usado para la medición local
gdf_near = (
    gdf_near.set_geometry("geometry_snap", crs=crs_metric)
            .to_crs(cfg.CRS_LATLON)
)

# 2) Columnas administrativas que traen sufijo "_dpa"
cols_admin_dpa = [c for c in gdf_near.columns if c.endswith("_dpa")]
rename_map = {c: c.replace("_dpa", "") for c in cols_admin_dpa}

# 3) Filas realmente emparejadas (CUT_COM_dpa ≠ NaN)
idx_fix = gdf_near.index[gdf_near["CUT_COM_dpa"].notna()]

# 4) Conservar la geometría original antes de la corrección
gdf_join.loc[idx_fix, "geometry_orig"] = gdf_join.loc[idx_fix, "geometry"]

# 5) Sustituir por la geometría corregida
gdf_join.loc[idx_fix, "geometry"] = gdf_near.loc[idx_fix, "geometry_snap"].values

# 6) Copiar atributos comunales oficiales
gdf_join.loc[idx_fix, list(rename_map.values())] = (
    gdf_near.loc[idx_fix, cols_admin_dpa]
            .rename(columns=rename_map)
            .values
)

# 7) Control final de nulos
print("\nPuntos sin comuna tras snap:", gdf_join["CUT_COM"].isna().sum())

In [ ]:
gdf_join.crs

### Revisión de puntos re ajustados

In [ ]:
# ---------------------------------------------------------------
# 1) Columnas administrativas que no deben contener nulos
# ---------------------------------------------------------------
cols_sin_na = ["CUT_REG", "CUT_PROV", "CUT_COM", "REGION",
               "PROVINCIA", "COMUNA", "SUPERFICIE"]

# ---------------------------------------------------------------
# 2) Conservar solo filas con asignación territorial completa
# ---------------------------------------------------------------
gdf_join_ok = gdf_join.dropna(subset=cols_sin_na).copy()

print(f"Filas antes  : {len(gdf_join):,}")
print(f"Filas después: {len(gdf_join_ok):,}")
gdf_join_ok.head()

In [ ]:
gdf_sin_nulos = gdf_join_ok[gdf_join_ok["geometry_orig"].notnull()]
gdf_sin_nulos.head()

In [ ]:
# Distancias de corrección efectivamente aplicadas (atributo de auditoría del snap)
gdf_near_not_na = gdf_near.loc[gdf_near["dist_m"].notnull(), "dist_m"]
print(f"Observaciones corregidas: {len(gdf_near_not_na):,}")
if len(gdf_near_not_na):
    print(gdf_near_not_na.describe())

Esta sección permite inspeccionar las observaciones efectivamente corregidas por el mecanismo de snap. La columna dist_m de gdf_near registra la distancia de corrección aplicada en cada caso, atributo de auditoría exigido por el Módulo II. Si el mecanismo no se activó, ambos conjuntos resultan vacíos.

## Tratamiento usuarios únicos

In [ ]:
# ────────────────────────────────────────────────────────────────
#        ANÁLISIS DE USUARIOS ÚNICOS  —  versión optimizada
# ────────────────────────────────────────────────────────────────

# 1) Conteo de registros por usuario
user_counts = gdf_join_ok[cfg.COL_USER].value_counts()

# 2) Métricas antes del filtrado
n_users_before  = user_counts.size
n_single_users  = (user_counts == 1).sum()
pct_single      = round(n_single_users / n_users_before * 100, 2)

print(f"Usuarios totales antes:            {n_users_before:,}")
print(f"Usuarios con un solo registro:     {n_single_users:,}  ({pct_single} %)")

# 3) Filtrar usuarios con ≥2 registros (sin crear lista intermedia)
keep_mask = gdf_join_ok[cfg.COL_USER].map(user_counts.ge(cfg.MIN_RECORDS_PER_USER))
rows_before = len(gdf_join_ok)
gdf_join_ok = gdf_join_ok.loc[keep_mask]

# 4) Compactar el índice sin copia profunda
gdf_join_ok.index = pd.RangeIndex(len(gdf_join_ok))

print(f"\nUsuarios totales después:         {gdf_join_ok[cfg.COL_USER].nunique():,}")
print(f"Filas antes:                      {rows_before:,}")
print(f"Filas después:                    {len(gdf_join_ok):,}")

# 5) Limpieza de objetos intermedios
del user_counts, n_users_before, n_single_users, pct_single, keep_mask, rows_before
_ = gc.collect()

## Guardado de dataframe pre procesado

In [ ]:
# Consolidación del registro depurado y asignado territorialmente (salida D2)

# 1) Partimos del GeoDataFrame resultante
df = gdf_join_ok.copy()

# 2) Quitar la columna de geometría original (auditoría del snap)
df = df.drop(columns=["geometry_orig"], errors="ignore")

# 3) Asegurar tipos útiles (preserva ceros a la izquierda en los códigos)
for c in ["CUT_REG", "CUT_PROV", "CUT_COM"]:
    if c in df.columns:
        df[c] = df[c].astype("string")

# 4) Asegurar datetime
if cfg.COL_TS in df.columns:
    df[cfg.COL_TS] = pd.to_datetime(df[cfg.COL_TS])

# 5) Orden canónico de columnas: identificación, tiempo, unidades DPA, geometría
cols = [
    cfg.COL_USER, cfg.COL_TS,
    "CUT_REG", "CUT_PROV", "CUT_COM",
    "REGION", "PROVINCIA", "COMUNA", "SUPERFICIE",
    "geometry",
]
df = df[[c for c in cols if c in df.columns]]

# 6) Persistencia opcional del registro depurado.
#    Nivel 2: deriva de la fuente telco y no forma parte de la entrega pública.
#    Descomente si desea conservar el intermedio en su propio entorno.
# df.to_parquet(cfg.CHILE_DIR / "pings_depurados.parquet", index=False)

del gdf_join_ok
_ = gc.collect()

df.head()

In [ ]:
df.dtypes

## Transiciones entre OAs

In [ ]:
# Cargar las OAs generadas en el cuaderno de preprocesamiento territorial
os.environ["OGR_GEOJSON_MAX_OBJ_SIZE"] = "0"  # 0 = sin límite
oas = gpd.read_file(cfg.OUTPUT_AREAS_GEOJSON, engine="pyogrio").to_crs(4326)
oas = oas[["OA_ID", "area_km2", "geometry"]].copy()

geod = Geod(ellps="WGS84")

In [ ]:
# Asegurar OA_ID como string de 5 dígitos
df = df.copy()
df["OA_ID"] = df["CUT_COM"].astype(str).str.zfill(5)

# Retener el mínimo necesario para construir las transiciones
df_od = df[[cfg.COL_USER, cfg.COL_TS, "OA_ID"]].rename(
    columns={cfg.COL_USER: "uid", cfg.COL_TS: "ts"}
)

# Restricción opcional a días hábiles, no aplicada en la corrida documentada.
# df_od = df_od[df_od["ts"].dt.dayofweek < 5]

df_od.head()

In [ ]:
# Orden temporal por usuario
df_od = df_od.sort_values(["uid", "ts"])

# OA siguiente y timestamp siguiente dentro del mismo usuario
df_od["oa_next"] = df_od.groupby("uid")["OA_ID"].shift(-1)
df_od["ts_next"] = df_od.groupby("uid")["ts"].shift(-1)

# Mantener solo cambios reales de OA
moves = df_od[(df_od["oa_next"].notna()) & (df_od["OA_ID"] != df_od["oa_next"])].copy()

# Filtro temporal opcional entre observaciones consecutivas.
# No se aplicó en la corrida documentada: las transiciones resultantes son
# cambios consecutivos de comuna, sin restricción de intervalo.
# dt = (moves["ts_next"] - moves["ts"]).dt.total_seconds() / 60.0   # minutos
# moves = moves[(dt >= 5) & (dt <= 479)]

flows = (moves
         .groupby(["OA_ID", "oa_next"])
         .size()
         .reset_index(name="flow")
         .rename(columns={"OA_ID": "origin", "oa_next": "destination"}))

# Retener solo OAs presentes en la capa territorial
valid = set(oas["OA_ID"])
flows = flows[flows["origin"].isin(valid) & flows["destination"].isin(valid)]

# Persistencia del producto relacional.
# Nivel 2: deriva de la fuente telco y no forma parte de la entrega pública.
flows.to_csv(cfg.FLOWS_CSV, index=False)  # columnas: origin,destination,flow
print("flows.csv generado:", cfg.FLOWS_CSV, "| pares OD:", len(flows))
flows.head()